<a href="https://colab.research.google.com/github/DezinTI/Data-Science-com-Pandas/blob/main/notebook_binario_universal_Zn_Zs_V_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook Universal para Bases Binárias Zn/Zs (Vfinal)
Neste trabalho vamos aprender:

Carregar qualquer base binária (.csv)
Criar/ajustar a coluna target para classificação binária
Verificar balanceamento
Treinar dois modelos (Regressão Logística & Random Forest)
Avaliar métricas, curvas e histogramas
Explorar pontos de corte para automação/manual

# 1. Importação de bibliotecas essenciais para Data Science e ML

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, auc,
    confusion_matrix
)
sns.set(style="whitegrid")

#Importa todas as bibliotecas necessárias para manipulação de dados, gráficos, processamento de texto/categorias, modelos de ML, métricas e visualização.

# 2. Upload e leitura robusta do CSV
Faz upload do arquivo, detecta separador (vírgula, tab, ponto-e-vírgula) e limpa aspas/BOM/espacos.

In [ ]:
from google.colab import files
import io

uploaded = files.upload()
filename = list(uploaded.keys())[0]
def read_csv_flex(filename):
    seps = [',', '\t', ';']
    for sep in seps:
        try:
            df = pd.read_csv(io.BytesIO(open(filename, 'rb').read()), sep=sep, quoting=3, encoding='utf-8', engine='python')
            df.columns = [c.replace('\ufeff','').replace('"','').replace("'",'').strip() for c in df.columns]
            if len(df.columns) >= 2:
                return df
        except Exception:
            continue
    with open(filename, encoding='utf-8') as f:
        lines = f.readlines()
    header = lines[0].strip().replace('"','').replace('\ufeff','')
    for sep in seps:
        if sep in header:
            header_cols = [h.strip() for h in header.split(sep)]
            data = [[d.strip() for d in line.strip().replace('"','').split(sep)] for line in lines[1:]]
            df = pd.DataFrame(data, columns=header_cols)
            if len(df.columns) >= 2:
                return df
    raise Exception("Não foi possível identificar o separador corretamente.")
df = read_csv_flex(filename)
print("Colunas disponíveis:", df.columns.tolist())
print(df.head())

#Faz upload do arquivo.
#Detecta automaticamente o separador (vírgula, tab, ponto-e-vírgula).
#Remove aspas, BOM e espaços das colunas.
#Retorna um DataFrame limpo para análise.
#O DataFrame df será usado em todas as células seguintes.

# 3. Limpeza geral da base e detecção do target binário
Converte target para 0/1, seja texto ou número.

In [ ]:
def limpar_base(df, col_target=None):
    df = df.copy()
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.replace('"','').str.replace("'",'').str.strip()
    if not col_target:
        possiveis = [c for c in df.columns if c.lower() in ['target','classe','class','label','spam','ham','is_spam','is_ham']]
        col_target = possiveis[0] if possiveis else df.columns[-1]
    target_map = {
        'spam': 1, 'ham': 0, 'yes': 1, 'no': 0,
        'true': 1, 'false': 0, 'sim': 1, 'nao': 0,
        'positive': 1, 'negative': 0, 'pos': 1, 'neg': 0,
        '1': 1, '0': 0, 1: 1, 0: 0
    }
    df[col_target] = df[col_target].astype(str).str.lower().str.strip().replace(target_map)
    df[col_target] = pd.to_numeric(df[col_target], errors='coerce')
    df[col_target] = df[col_target].fillna(0).astype(int)
    vals = set(df[col_target].unique())
    if not vals.issubset({0,1}):
        raise ValueError(f"Coluna target não está binária! Valores encontrados: {vals}")
    return df, col_target
df, col_target = limpar_base(df)
print("Target limpo! Valores únicos:", df[col_target].unique(), df[col_target].dtype)
print(df[col_target].value_counts(normalize=True)*100)

#Limpa todas as colunas object: tira aspas, espaços extras.
#Detecta a coluna target (pode ser chamada de target, classe, label, spam, etc).
#Converte target para 0/1, mesmo se vier como texto (ex: spam/ham, yes/no).
#Garante que o target está binário e pronto para ML

# 4. Verificação do balanceamento percentual da base
Mostra visualmente o balanceamento da base binária.

In [ ]:
sns.countplot(data=df, x=col_target, palette="Set2")
plt.title("Balanceamento do target")
plt.xlabel("target")
plt.ylabel("Contagem")
plt.show()

#Plota gráfico mostrando a distribuição dos valores da coluna target.
#Permite visualizar se há desbalanceamento entre as classes (ex: 80%/20%).
#Importante para escolha de métricas (ex: AUC-PR é melhor em bases desbalanceadas).

# 5. Limpeza adicional
Remove colunas muito vazias/repetidas e linhas sem target.

In [ ]:
def limpeza_extra(df, col_target):
    lim = 0.9 * len(df)
    cols_to_drop = [c for c in df.columns if df[c].isnull().sum() > lim]
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)
        print("Colunas removidas por excesso de nulos:", cols_to_drop)
    df = df.loc[:,~df.columns.duplicated()]
    df = df[df[col_target].notnull()]
    return df
df = limpeza_extra(df, col_target)

#Remove colunas com mais de 90% de valores nulos.
#Remove colunas repetidas.
#Remove linhas sem valor para o target.
#Mantém apenas os exemplos válidos para treino/teste.

# 6. Detecta tipos de features (numéricas, categóricas e texto)
Separa automaticamente colunas numéricas, categóricas e texto.

In [ ]:
col_text = None
for col in df.columns:
    if col == col_target: continue
    if df[col].dtype == 'object' and df[col].astype(str).str.len().mean() > 15:
        col_text = col
        break
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include='object').columns.tolist()
for lst in (numerical_cols, categorical_cols):
    if col_target in lst: lst.remove(col_target)
if col_text and col_text in categorical_cols: categorical_cols.remove(col_text)
print("Numéricas:", numerical_cols)
print("Categóricas:", categorical_cols)
print("Texto longa:", col_text)
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
transformers = []
if numerical_cols:
    transformers.append(('num', SimpleImputer(strategy='mean'), numerical_cols))
if categorical_cols:
    transformers.append(('cat', OneHotEncoder(handle_unknown='ignore', sparse=False), categorical_cols))
if col_text:
    transformers.append(('txt', TfidfVectorizer(), col_text))
preprocessor = ColumnTransformer(transformers, remainder='drop')

#Identifica colunas numéricas, categóricas e texto longo.
#Prepara listas separadas para cada tipo.
#Monta um ColumnTransformer para pré-processar automaticamente cada tipo de dado.
#Permite que o pipeline trate cada tipo de coluna do jeito correto.

# 7. Split treino/teste estratificado
Divide base em treino e teste mantendo proporção das classes.

In [ ]:
X = df.drop(columns=[col_target])
y = df[col_target].values
try:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
except Exception as e:
    print("Split estratificado falhou:", e)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print("Shapes: X_train", X_train.shape, "X_test", X_test.shape)
print("Counts treino:", pd.Series(y_train).value_counts())

#Divide a base em treino (70%) e teste (30%), mantendo proporção das classes.
#Produz variáveis X_train, X_test, y_train, y_test usadas nos modelos e métricas seguintes.

# 8. Pipeline dos modelos (Logistic Regression e Random Forest)
Pré-processamento e modelo juntos.

In [ ]:
pipe_lr = Pipeline([('preproc', preprocessor), ('clf', LogisticRegression(max_iter=1000))])
pipe_rf = Pipeline([('preproc', preprocessor), ('clf', RandomForestClassifier(random_state=42))])

#Cria pipeline para cada modelo, incluindo pré-processamento e o classificador.
#Garante que o mesmo tratamento é aplicado no treino e no teste.
#Pipelines serão usados nas células de treino, avaliação e grid search.

# 9. Treinamento e comparação dos dois modelos
Compara desempenho entre Logistic Regression e Random Forest.

In [ ]:
pipe_lr.fit(X_train, y_train)
y_pred_lr = pipe_lr.predict(X_test)
print("\nLogistic Regression:")
print("Acurácia:", accuracy_score(y_test, y_pred_lr), "F1:", f1_score(y_test, y_pred_lr))
pipe_rf.fit(X_train, y_train)
y_pred_rf = pipe_rf.predict(X_test)
print("\nRandom Forest:")
print("Acurácia:", accuracy_score(y_test, y_pred_rf), "F1:", f1_score(y_test, y_pred_rf))

#Treina ambos os modelos (Logistic Regression e Random Forest) nos dados de treino.
#Faz predição nos dados de teste.
#Exibe acurácia e F1 dos dois para comparar rapidamente.
#Resultados serão usados para escolher o melhor modelo para grid search e gráficos.

# 10. GridSearchCV para RandomForest (otimização de hiperparâmetros)
Busca melhores hiperparâmetros para RF.

In [ ]:
min_count = np.min(pd.Series(y_train).value_counts())
if min_count >= 2:
    cv = 3 if min_count >= 3 else 2
    param_grid = {
        'clf__n_estimators': [50, 100],
        'clf__max_depth': [None, 5, 10],
        'clf__min_samples_split': [2, 5]
    }
    print(f"Rodando GridSearch cv={cv} ...")
    grid = GridSearchCV(pipe_rf, param_grid, cv=cv, scoring='f1', n_jobs=-1)
    grid.fit(X_train, y_train)
    print("Melhores parâmetros:", grid.best_params_)
    melhor_modelo = grid.best_estimator_
else:
    print("GridSearch pulado (poucas amostras por classe).")
    melhor_modelo = pipe_rf

#Roda busca automática dos melhores hiperparâmetros para Random Forest.
#Usa validação cruzada (cv) adaptada ao tamanho da menor classe.
#Retorna o melhor modelo encontrado (melhor_modelo), que será usado nas métricas e gráficos seguintes.

# 11. Métricas finais e matriz de confusão
Avalia desempenho e mostra FP/FN.

In [ ]:
y_prob = melhor_modelo.predict_proba(X_test)[:, 1]
y_pred = melhor_modelo.predict(X_test)
print("\nMétricas finais:")
print("Acurácia:", accuracy_score(y_test, y_pred))
print("Precisão:", precision_score(y_test, y_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_pred, zero_division=0))
print("F1-score:", f1_score(y_test, y_pred, zero_division=0))
print("AUC-ROC:", roc_auc_score(y_test, y_prob))
print("AUC-PR:", average_precision_score(y_test, y_prob))
cm = confusion_matrix(y_test, y_pred)
print("\nMatriz de confusão:")
print(cm)
total = np.sum(cm)
falsos_positivos = cm[0,1]
falsos_negativos = cm[1,0]
print(f"Falsos Positivos (FP): {falsos_positivos} ({(falsos_positivos/total)*100:.1f}%)")
print(f"Falsos Negativos (FN): {falsos_negativos} ({(falsos_negativos/total)*100:.1f}%)")

#Avalia o modelo final com métricas: acurácia, precisão, recall, F1, AUC-ROC, AUC-PR.
#Calcula a matriz de confusão (VP, VN, FP, FN).
#Exibe número e percentual de falsos positivos e falsos negativos.
#Usa y_test (verdadeiros) e y_pred (previsão do modelo final).

# 12. Curva ROC e Curva Precision-Recall (PR)
Visualização das curvas para avaliação dos modelos.

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(5,4))
plt.plot(fpr, tpr, label=f'ROC AUC={roc_auc:.2f}')
plt.plot([0,1],[0,1],'k--', linewidth=0.7)
plt.xlabel('FPR (Taxa de Falsos Positivos)')
plt.ylabel('TPR (Recall)')
plt.title('Curva ROC')
plt.legend()
plt.tight_layout()
plt.show()

prec_vals, rec_vals, _ = precision_recall_curve(y_test, y_prob)
pr_auc = auc(rec_vals, prec_vals)
plt.figure(figsize=(5,4))
plt.plot(rec_vals, prec_vals, label=f'PR AUC={pr_auc:.2f}')
plt.xlabel('Recall')
plt.ylabel('Precisão')
plt.title('Curva Precision-Recall')
plt.legend()
plt.tight_layout()
plt.show()

#Plota curva ROC: compara taxa de verdadeiros positivos (TPR/Recall) e taxa de falsos positivos (FPR) para vários thresholds.
#Calcula com roc_curve(y_test, y_prob) e auc(fpr, tpr).
#Depende de y_test e y_prob.
#Plota curva PR: mostra relação entre precisão e recall para vários thresholds.
#Calcula com precision_recall_curve(y_test, y_prob) e auc(recall, precision).
#Também depende de y_test e y_prob.

# 13. Função de avaliação de thresholds (ponto de corte)
threshold - ponto de corte da probabilidade para decidir negativo, positivo ou manual.
Para alterar os thresholds, mude os valores em 'for neg in [0.2, 0.3]:' e 'for pos in [0.7, 0.8]:' (ex: [0.1, 0.2, 0.3], [0.7, 0.8, 0.9])

In [ ]:
def avaliar_thresholds(y_true, y_prob, th_neg=0.3, th_pos=0.7):
    th_neg = float(th_neg); th_pos = float(th_pos)
    assert 0 <= th_neg < th_pos <= 1
    preds = []
    for p in y_prob:
        if p <= th_neg:
            preds.append(0)
        elif p >= th_pos:
            preds.append(1)
        else:
            preds.append(-1)
    dfc = pd.DataFrame({"Prob": y_prob, "Target": y_true, "Pred": preds})
    df_auto = dfc[dfc['Pred'] != -1]
    if len(df_auto) > 0:
        acc = accuracy_score(df_auto["Target"], df_auto["Pred"])
        prec = precision_score(df_auto["Target"], df_auto["Pred"], zero_division=0)
        rec = recall_score(df_auto["Target"], df_auto["Pred"], zero_division=0)
        f1 = f1_score(df_auto["Target"], df_auto["Pred"], zero_division=0)
        cm = confusion_matrix(df_auto["Target"], df_auto["Pred"])
        fp = cm[0,1] if cm.shape[1] > 1 else 0
        fn = cm[1,0] if cm.shape[1] > 1 else 0
        pct_fp = (fp / len(df_auto))*100 if len(df_auto) > 0 else 0
        pct_fn = (fn / len(df_auto))*100 if len(df_auto) > 0 else 0
    else:
        acc = prec = rec = f1 = fp = fn = pct_fp = pct_fn = 0.0
    total = len(dfc)
    pct_neg = (dfc['Pred']==0).sum()/total*100
    pct_pos = (dfc['Pred']==1).sum()/total*100
    pct_manual = (dfc['Pred']==-1).sum()/total*100
    print(f"Corte {th_neg:.2f}/{th_pos:.2f} -> auto_negativo {pct_neg:.1f}%, auto_positivo {pct_pos:.1f}%, manual {pct_manual:.1f}%")
    print(f"Métricas (automáticos): acurácia {acc:.3f}, precisão {prec:.3f}, recall {rec:.3f}, f1 {f1:.3f}")
    print(f"Falsos Positivos (FP): {fp} ({pct_fp:.1f}%) | Falsos Negativos (FN): {fn} ({pct_fn:.1f}%)")
    print(f"Total de exemplos automáticos: {len(df_auto)} | Total de exemplos manuais: {(dfc['Pred']==-1).sum()} ({pct_manual:.1f}%)")
    print('-'*40)
    return dfc
for neg in [0.2, 0.3]:
    for pos in [0.7, 0.8]:
        avaliar_thresholds(y_test, y_prob, neg, pos)

#Define thresholds para automação de decisões: corte negativo, positivo, manual.
#Para cada exemplo do teste, se prob <= corte_negativo: classifica como negativo; se prob >= corte_positivo: positivo; entre eles: manual.
#Calcula e mostra: acurácia, precisão, recall, F1, FP, FN, percentual de exemplos automáticos/manual.
#Para testar diferentes thresholds, altere os valores em for neg in ... e for pos in ....

# 14. Gráfico dos bins de probabilidade
Visualiza distribuição dos exemplos para justificar thresholds.
Para alterar o step (largura dos bins), mude o valor de 'step' abaixo. Sugestão: 0.05, 0.1, 0.2

In [ ]:
step = 0.1
bins = np.arange(0, 1 + step, step)
df_plot = pd.DataFrame({'prob_pos': y_prob, 'target': y_test})
df_plot['bin'] = pd.cut(df_plot['prob_pos'], bins=bins, include_lowest=True)
pop_total = df_plot['bin'].value_counts(normalize=True).sort_index() * 100
pop_pos = df_plot[df_plot['target']==1]['bin'].value_counts(normalize=True).sort_index() * 100
pop_pos = pop_pos.reindex(pop_total.index, fill_value=0)
labels = [f"{int(i.left*100)}-{int(i.right*100)}%" for i in pop_total.index]
x = np.arange(len(labels)); width=0.35
plt.figure(figsize=(10,4))
plt.bar(x-width/2, pop_total.values, width=width, color='tab:blue', label='Pop Geral')
plt.bar(x+width/2, pop_pos.values, width=width, color='tab:red', label='Positivos')
plt.xticks(x, labels, rotation=45)
plt.ylabel('% populacional')
plt.title('Distribuição de probabilidades por bin')
plt.legend(); plt.tight_layout(); plt.show()

#Visualiza distribuição dos exemplos (todos e positivos) por faixa de probabilidade.
#Permite justificar escolha dos thresholds.
#Para alterar o step (largura dos bins), modifique o valor de step (ex: 0.05, 0.1, 0.2).
#Exemplo: step = 0.1 cria bins de 10% de largura; step = 0.05 cria bins de 5%.